## Import needed libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import spacy
import string
import sklearn
from spacy import displacy
from pathlib import Path
from spacy import displacy
import os
from collections import Counter
import nltk
# Download the stopwords and tokenizer from nltk
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from google.colab import drive
drive.mount('/content/drive')


# Load the English language model
nlp = spacy.load('en_core_web_sm')

## Downloading class dataset

In [ ]:
# stored in my google drive
!unzip -q /content/drive/MyDrive/Colab\ Notebooks/datasets/BBC\ News\ Summary-kaggle-v2.zip

## Importing and converting data to panda data frame

In [ ]:
classes = os.listdir('/content/BBC News Summary/News Articles')
Articles_dir = '/content/BBC News Summary/News Articles/'

# initializing feature column data types
articles = []
file_arr = []
is_sport = []
is_biz = []
is_ent = []
is_politics = []
is_tech =[]
labels = []
for cls in classes:
    files = os.listdir(Articles_dir + cls)

    for file in files:
        article_file_path = Articles_dir + cls + '/' + file

        # Starting to append values for the different feature columns
        # extracting data from files
        try:
            with open (article_file_path,'r') as f:
                articles.append('.'.join([line.rstrip() for line in f.readlines()]))

            file_arr.append(cls + '/' + file)
            labels.append(cls)
            match cls:
              case "sport":
                is_sport.append(1)
                is_biz.append(0)
                is_ent.append(0)
                is_politics.append(0)
                is_tech.append(0)
              case "tech":
                is_sport.append(0)
                is_biz.append(0)
                is_ent.append(0)
                is_politics.append(0)
                is_tech.append(1)
              case "business":
                is_sport.append(0)
                is_biz.append(1)
                is_ent.append(0)
                is_politics.append(0)
                is_tech.append(0)
              case "entertainment":
                is_sport.append(0)
                is_biz.append(0)
                is_ent.append(1)
                is_politics.append(0)
                is_tech.append(0)
              case "politics":
                is_sport.append(0)
                is_biz.append(0)
                is_ent.append(0)
                is_politics.append(1)
                is_tech.append(0)
        except:
            pass

df = pd.DataFrame({'File_path':file_arr,'Articles text': articles, 'label': labels, 'Business': is_biz, 'Entertainment': is_ent, 'Politics': is_politics, 'Sport': is_sport, 'Tech': is_tech})
df.head()

## Data Cleaning and Appending


Preprocessing data is essentially to getting optimal model results. I used a modular approach and created a function to process each "Articles text" row in the data.  I first began with tokenizing the text which better helped identify sets of words and distinguish puntuation more properly. I also lowered all the words and removing the following: punctuation, stopwords, and duplicate words. I was hesitant with removing duplicate words, but I was satisfied with the results I was able to produce.  

The process_text function returns the clean data as "processed_text" and gets added as a new data column. I also reused another function called get_token_count, to create new columns detailing count of tokens for the original Articles text and processed_text. This gave me great initial insight to see how much information was reduced, and to see how much was relevent.

In [ ]:
def preprocess_text(text):
    # Convert text to lowercase
    words = word_tokenize(text)

    # Convert words to lowercase
    words = [word.lower() for word in words]

    # Remove punctuation from words
    words = [word for word in words if word.isalnum()]

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]

    # Remove duplicate words
    unique_words = list(dict.fromkeys(words))

    # Join the words back into a string
    text = ' '.join(unique_words)

    return text

def get_token_count(text):
    words = word_tokenize(text)
    return len(words)


# Apply the function to the DataFrame column
df['processed_text'] = df['Articles text'].apply(preprocess_text)
df['processed_text_token_count'] = df['processed_text'].apply(get_token_count)
df['Articles_text_token_count'] = df['Articles text'].apply(get_token_count)

display(df.head())
print("sample of cleaned text \n", df['processed_text'][0])

## EDA of Articles Text using NER and POS visualization

I was able to explore the data, and how well the processed text preformed with these visualizers. Although, the data is cleaned some context seems to be missing. At a high level I am able to interpret a who, what, and when quickly using the viualizer.

The part of speech visualizer is much easier to read when processing text this way. The visual is much cleaner and easier to follow with less words in the prompt. All the parts of speech are clearly labeled show dependencies.


*NOTE:* This is the correct code for using these visualizations and I can view them in my local notebook / colab but I can't make them visible when using jupyter nbconvert.

In [ ]:
doc = nlp(df['processed_text'][0])

# produce spacy ent results
displacy.render(doc, style='ent', minify=True, jupyter=True)

# produce dependency and labeling results
options = {"compact": True, "bg": "#09a3d5","color": "white", "font": "Source Sans Pro", "distance": 70}
displacy.render(doc, style="dep",minify=True,jupyter=True, options=options)


#### Initial Inisight on Label Distribution

In [ ]:
labels = ['Business', 'Entertainment', 'Politics', 'Sport', 'Tech']
counts = [df[label].sum() for label in labels]

# Creating a visualization for the distribution of the labels.
plt.figure(figsize=(7, 4))
plt.barh(labels, counts, color='skyblue', label="hi")
for index, value in enumerate(counts):
    plt.text(value, index, str(value))
plt.xlabel('Count')
plt.ylabel('Labels')
plt.title('Distribution of Labels')
plt.show()

## Train, test split

This dataset was complex because it dealt with multiple labels as features
compared to just one, like in the lab session. One of the biggest issues was understanding how to reshape the values prior to training so that the y values could produce results for multiple columns and not just one.

This was done by reshapeing the processed_text to match the shape of the multiple columns data (y). This allowed me to properly train test and split the data with the corresponding values of the multiple columns. After splitting I had to convert back to 1d vector for it to be compatable with TF-IDF Vectorization function.

In [ ]:
X = df['processed_text'].values
y = df[['Business', 'Entertainment', 'Politics', 'Sport', 'Tech']].values

# Convert X to a NumPy array of shape (n_samples, 1) to match y's shape of (n_samples, n_labels)
# y.shape (2224, 5)
X = np.array(X).reshape(-1, 1)

# Step 2: Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print("Label distribution in test set:", np.sum(y_test, axis=0))
print("Label distribution in entire dataset:", np.sum(y, axis=0))

# Reshape to reshape nparrays from 2d to 1d for vectorization
X_train = X_train.reshape(-1)
X_test = X_test.reshape(-1)


## Apply TF-IDF Vectorization


Term Frequency-Inverse Document Frequency (TF-IDF), is used to convert text data into a numerical format, preserving important textual characteristics in a way that can be easily used by algorithms.

In [ ]:
# Applying vectorizer on data
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

display(X_test_tfidf)

## Evaluate Metrics Function Created


The following classification metrics were caputed to get ingsight:


1. Accuracy: Correct predictions over the total number of instances.
2. Precision: Measures the proportion of true positive predictions among the instances that the model predicted as positive.
3. Recall: Measures the proportion of true positive predictions among the actual positive instances. Measures ability to identify all relevant instances.4. F1 Score: Considers both false positives and false negatives, and is the mean score of precision and recall.
5. AUC-ROC Score: Tells how much the model is capable of distinguishing between classes.


A classification report was also generated to get a macro view of the performance including all the feature labels.

Using a confusion matrix I was able to better view performace of the feature columns. The different values represent the count of true negatives (at 0,0), false negatives (at 1,0), true positives (at 1,1), and false positives (at 0,1) in the matrix.

In [ ]:
def eval_metrics(X_test_tfidf, y_test, y_pred_tfidf, clf):
  # Evaluate using classification metrics
  accuracy_tfidf = accuracy_score(y_test, y_pred_tfidf)
  precision_tfidf = precision_score(y_test, y_pred_tfidf, average='weighted')
  recall_tfidf = recall_score(y_test, y_pred_tfidf, average='weighted')
  f1_score_tfidf = f1_score(y_test, y_pred_tfidf, average='weighted')

  # Using predict_proba which peformed similar results to decision_function()
  roc_auc_tfidf = roc_auc_score(y_test, clf.predict_proba(X_test_tfidf))

  # Display the classification metrics
  print(f"Accuracy: {accuracy_tfidf:.2}")
  print(f"Precision: {precision_tfidf:.2}")
  print(f"Recall: {recall_tfidf:.2}")
  print(f"F1 Score: {f1_score_tfidf:.2}")
  print(f"AUC-ROC Score: {roc_auc_tfidf:.2}")

  labels = ['Business', 'Entertainment', 'Politics', 'Sport', 'Tech']
  # Report on classification metrics for each of the features
  print(classification_report(y_test, y_pred_tfidf, target_names=labels))

  for i, class_label in enumerate(labels):
      print(f"Confusion matrix for {class_label}:")
      print(confusion_matrix(y_test[:, i], y_pred_tfidf[:, i]))

## Apply SVM and Random Forest

OneVsRestClassifier is a simple and effective method for adapting binary classifiers for multi-class or multi-label problems, but it may not always be the most efficient in terms of computation and memory usage, especially when the number of classes is large.

In addition to its computational efficiency (only n_classes classifiers are needed), one advantage of this approach is its interpretability. Since each class is represented by one and one classifier only, it is possible to gain knowledge about the class by inspecting its corresponding classifier (sklearn.multiclass.OneVsRestClassifier?).

In [ ]:
# Create a OneVsRestClassifier instance with a Support Vector Machine (SVM) classifier
svm_classifier = SVC(kernel='linear', random_state=42, probability=True)
clf = OneVsRestClassifier(svm_classifier)
# Fit the classifier
clf.fit(X_train_tfidf, y_train)
# Predictions on the test set
y_pred_tfidf = clf.predict(X_test_tfidf)

In [ ]:
print("Results using TF-IDF Vectorization and Support Vector Machine (SVM) Classifier:")
eval_metrics(X_test_tfidf, y_test, y_pred_tfidf, clf)

In [ ]:
# Create a OneVsRestClassifier instance with a RandomForestClassifier estimator
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
clf = OneVsRestClassifier(rf_classifier)

# Fit the classifier
clf.fit(X_train_tfidf, y_train)

# Make predictions
y_pred_tfidf_rfc = clf.predict(X_test_tfidf)

In [ ]:
print("Results using TF-IDF Vectorization and Random Forest Classifier:")
eval_metrics(X_test_tfidf, y_test, y_pred_tfidf_rfc, clf)

## Conclusion

From the metrics, I could see that the SVM algorithm was more accurate, had better recall, and a better F1 Score than the random forest model. Both algorithms agreed and classified sports with the best metrics.

In doing this assignment again, I would explore more pre processing techniques and run tests with these algorithms by altering which tasks of preprocessing I do.

## References:

https://scikit-learn.org/stable/modules/generated/sklearn.multiclass.OneVsRestClassifier.html

https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html

## Exporting to pdf

In [ ]:
!jupyter nbconvert --to pdf ~/